# Feature Selection & Dimensionality Optimization

In [1]:
# imports 

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from catboost import CatBoostClassifier

import shap 


In [6]:
# Create Engineered Dataset

from src.preprocessing import PreprocessConfig, preprocess_train_test
from src.features import FeatureConfig, engineer_train_test_features

# Load raw data
TRAIN_PATH = "D:\\dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n\\Train.csv"
TEST_PATH  = "D:\\dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n\\Test.csv"

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

# Initiliaze configs
pre_cfg = PreprocessConfig(
    id_col="ID",
    target_col="Target"
)

feat_cfg = FeatureConfig(
    id_col="ID",
    target_col="Target"
)

#run preprocessing
train_clean, test_clean = preprocess_train_test(
    train_raw,
    test_raw,
    pre_cfg,
    for_model="lightgbm"  # We use "lightgbm" here because the output is fully numeric, which is easier for: correlation analysis, SHAP & feature importance
)

# Run feature engineering
train_fe, test_fe = engineer_train_test_features(
    train_clean,
    test_clean,
    feat_cfg,
    collapse_rare_for_non_catboost=True
)

In [7]:
# Save the Engineered Dataset

import os

os.makedirs("data/processed", exist_ok=True)

train_fe.to_csv("data/processed/train_engineered.csv", index=False)
test_fe.to_csv("data/processed/test_engineered.csv", index=False)

print("Engineered datasets saved.")

Engineered datasets saved.


In [8]:
# Load feature selection

train_df = pd.read_csv("data/processed/train_engineered.csv")

TARGET = "Target"
ID = "ID"

X = train_df.drop(columns=[TARGET, ID])
y = train_df[TARGET]

feature_names = X.columns.tolist()

print("Feature count:", len(feature_names))

Feature count: 117
